In [35]:
import pandas as pd
import numpy as np


terrorism = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\Global Terrorism Dataset\globalterrorismdb_0718dist.csv", encoding="Windows-1252")

C:\Users\HP\AppData\Local\Temp\ipykernel_10036\551476585.py:5: DtypeWarning: Columns (4,6,31,33,61,62,63,76,79,90,92,94,96,114,115,121) have mixed types. Specify dtype option on import or set low_memory=False.
  terrorism = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\Global Terrorism Dataset\globalterrorismdb_0718dist.csv", encoding="Windows-1252")


In [36]:
#Run this cell only once. The conversion of unknown values to NaN leads to them being dropped on a second run.
import pathlib
print(terrorism["gname"].unique().shape[0])
terrorism["gname"] = terrorism["gname"].str.strip()
terrorism["gname"] = terrorism["gname"].str.lower()
ranks = terrorism.groupby("gname")["eventid"].count().sort_values(ascending=False)
ranks = ranks[ranks>30]
terrorism = terrorism[terrorism["gname"].isin(ranks.index)]
terrorism["gname"] = terrorism["gname"].replace(to_replace={"unknown":pd.NA, "maoists":"communist party of india - maoist (cpi-maoist)", "palestinian extremists":"palestine liberation organization (plo)", "palestinians": "palestine liberation organization (plo)", "muslim militants":"muslim extremists", "muslim separatists":"muslim extremists","muslim fundamentalists":"muslim extremists","muslim rebels":"muslim extremists", "muslims":"muslim extremists", "islamic state of iraq (isi)":"islamic state of iraq and the levant (isil)" })

ranks = terrorism.groupby("gname")["eventid"].count().sort_values(ascending=False)
str_ranks = ranks.to_string()

with open(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\ranks.txt", "w") as file_rank:
    file_rank.write(str_ranks)

terrorism.shape

3537


(170274, 135)

In [37]:

collist = [col for col in terrorism.columns if any(w in col.lower() for w in ["target", "ransom", "targ", "corp", "natlty", "guncertain", "claimmode", "weap", "ishostkid", "scite", "hostkidoutcome", "prop", "nperp"])] 
collist+= ["attacktype2_txt", "attacktype2", "attacktype3", "attacktype3_txt", "gsubname2", "gsubname3", "gname2", "gname3", "dbsource", "addnotes", "nkillus", "nwoundus", "nkillter", "nwoundte","location" ]
terrorism = terrorism.drop(columns=collist)

terrorism = terrorism.drop( columns= [col for col in terrorism.columns if ((terrorism[col].isna().sum()/len(terrorism[col]))>0.85)])
terrorism.columns

Index(['eventid', 'iyear', 'imonth', 'iday', 'extended', 'country',
       'country_txt', 'region', 'region_txt', 'provstate', 'city', 'latitude',
       'longitude', 'specificity', 'vicinity', 'summary', 'crit1', 'crit2',
       'crit3', 'doubtterr', 'alternative', 'alternative_txt', 'multiple',
       'success', 'suicide', 'attacktype1', 'attacktype1_txt', 'gname',
       'motive', 'individual', 'claimed', 'nkill', 'nwound', 'INT_LOG',
       'INT_IDEO', 'INT_MISC', 'INT_ANY'],
      dtype='object')

In [38]:
terrorism.isna().sum()

eventid                 0
iyear                   0
imonth                  0
iday                    0
extended                0
country                 0
country_txt             0
region                  0
region_txt              0
provstate             346
city                  431
latitude             4145
longitude            4146
specificity             5
vicinity                0
summary             60111
crit1                   0
crit2                   0
crit3                   0
doubtterr               1
alternative        142822
alternative_txt    142822
multiple                1
success                 0
suicide                 0
attacktype1             0
attacktype1_txt         0
gname               82782
motive             123271
individual              0
claimed             60102
nkill                9578
nwound              15275
INT_LOG                 0
INT_IDEO                0
INT_MISC                0
INT_ANY                 0
dtype: int64

In [39]:
#Date Parsing code
terrorism["date"] = terrorism.iyear.astype(str) + "/" + terrorism.imonth.astype(str) + "/" + terrorism.iday.astype(str)
terrorism["date"] = pd.to_datetime(terrorism["date"], format=f"%Y/%#m/%#d", errors="coerce")
print(len(terrorism["date"]))
print(terrorism["date"].isna().sum())

terrorism = terrorism.dropna(subset="date")
terrorism = terrorism.drop(columns=["imonth", "iyear", "iday"])


170274
766


In [40]:
cities = pd.read_csv(f"Global Terrorism Dataset/worldcitiespop.csv")
cities["City"] = cities["City"].str.lower()
cities["City"] = cities["City"].str.strip()
print(cities.head())
cities = cities.drop_duplicates(subset="City")
terrorism["city"] = terrorism["city"].str.lower()
terrorism["city"] = terrorism["city"].str.strip()
terrorism = pd.merge(terrorism, cities, how="left", left_on="city", right_on="City")

terrorism["latitude"] = terrorism.latitude.fillna(terrorism.Latitude)
terrorism["longitude"] = terrorism.longitude.fillna(terrorism.Longitude)


C:\Users\HP\AppData\Local\Temp\ipykernel_10036\4142167283.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  cities = pd.read_csv(f"Global Terrorism Dataset/worldcitiespop.csv")


  Country        City  AccentCity Region  Population   Latitude  Longitude
0      ad       aixas       Aixàs    6.0         NaN  42.483333   1.466667
1      ad  aixirivali  Aixirivali    6.0         NaN  42.466667   1.500000
2      ad  aixirivall  Aixirivall    6.0         NaN  42.466667   1.500000
3      ad   aixirvall   Aixirvall    6.0         NaN  42.466667   1.500000
4      ad    aixovall    Aixovall    6.0         NaN  42.466667   1.483333


In [41]:
terrorism["Latitude"].isna().sum() *100 / len(terrorism["Latitude"])
terrorism["Latitude"].head()
terrorism["longitude"].isna().sum()
terrorism = terrorism.drop(columns=["Country","City","AccentCity","Region","Population","Latitude","Longitude"])

In [ ]:
terrorism.groupby(["country_txt", "gname"])[[ "nkill", "nwound"]].agg(["sum", "mean", "count"])

eventid  \
                                                                      sum   
country_txt gname                                                           
Afghanistan al-fatah                                         200909150005   
            al-qaida                                        4005921170062   
            guerrillas                                      1194763230050   
            gunmen                                          1810000480236   
            haqqani network                                16899000691575   
...                                                                   ...   
Zimbabwe    anti-government extremists                       396513370015   
            guerrillas                                       198103160004   
            gunmen                                           200208290007   
            mozambique national resistance movement (mnr)   1591428200036   
            right-wing extremists                            199004280010   

                                                                               \
                                                                   mean count   
country_txt gname                                                               
Afghanistan al-fatah                                       2.009092e+11     1   
            al-qaida                                       2.002961e+11    20   
            guerrillas                                     1.991272e+11     6   
            gunmen                                         2.011112e+11     9   
            haqqani network                                2.011786e+11    84   
...                                                                 ...   ...   
Zimbabwe    anti-government extremists                     1.982567e+11     2   
            guerrillas                                     1.981032e+11     1   
            gunmen                                         2.002083e+11     1   
            mozambique national resistance movement (mnr)  1.989285e+11     8   
            right-wing extremists                          1.990043e+11     1   

                                                           nkill            \
                                                             sum      mean   
country_txt gname                                                            
Afghanistan al-fatah                                         0.0  0.000000   
            al-qaida                                        89.0  4.450000   
            guerrillas                                      12.0  2.000000   
            gunmen                                          13.0  1.444444   
            haqqani network                                606.0  7.390244   
...                                                          ...       ...   
Zimbabwe    anti-government extremists                       7.0  3.500000   
            guerrillas                                       2.0  2.000000   
            gunmen                                           0.0  0.000000   
            mozambique national resistance movement (mnr)   37.0  4.625000   
            right-wing extremists                            0.0  0.000000   

                                                                 nwound  \
                                                          count     sum   
country_txt gname                                                         
Afghanistan al-fatah                                          1     2.0   
            al-qaida                                         20   271.0   
            guerrillas                                        6    73.0   
            gunmen                                            9     1.0   
            haqqani network                                  82  1009.0   
...                                                         ...     ...   
Zimbabwe    anti-government extremists                        2     0.0   
         